# CrewAI: Orchestrating Collaborative Agent Systems

**Objective:** This notebook provides a hands-on guide to CrewAI, a framework designed for engineering sophisticated multi-agent systems. You will learn how to create a "crew" of autonomous AI agents that collaborate to achieve complex goals.

**Target Audience:** Software engineers attending the AI-Driven Software Engineering Program.

**Core Philosophy:** CrewAI's strength lies in its intuitive, high-level abstractions for defining agents with specific roles, assigning them tasks, and defining the workflow (process) they should follow. It simplifies the creation of systems where multiple agents work together, each contributing its specialized skills.

## 1. Setup

First, we'll install the necessary libraries. CrewAI uses LangChain components under the hood. We will also install the `tavily-python` library directly to create a custom search tool, which helps avoid complex dependency issues.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GOOGLE_API_KEY") or not os.getenv("TAVILY_API_KEY"):
    print("ERROR: OPENAI_API_KEY or TAVILY_API_KEY not found. Please check your .env file.")

## 2. Foundational Crew: The Two-Agent Team

Our first example will be a simple, two-agent crew designed to plan a trip. This will introduce the three core concepts of CrewAI:

-   **`Agent`**: A role-playing entity with a specific `role`, `goal`, and `backstory`.
-   **`Task`**: A specific unit of work to be performed by an agent.
-   **`Crew`**: A collection of agents and tasks, along with a defined `process` for execution.

In [2]:
from crewai import Agent, Task, Crew, Process, LLM

# Use a LangChain model as the LLM for the agents
llm = LLM(
    model="google/gemini-2.5-pro",
    api_key=os.getenv("GOOGLE_API_KEY"),

)

# Agent 1: The Travel Expert
travel_agent = Agent(
    role='Expert Travel Agent',
    goal='Create a detailed, 3-day itinerary for a trip.',
    backstory='You have 20 years of experience in luxury travel planning and know all the hidden gems.',
    verbose=True,
    llm=llm
)

# Agent 2: The Local Food Critic
food_critic = Agent(
    role='Local Food Critic',
    goal='Recommend the best, most authentic local restaurants for the trip.',
    backstory='You are a famous food blogger who lives in the destination city and is an expert on its culinary scene.',
    verbose=True,
    llm=llm
)

# Task 1: Plan the Itinerary (for the Travel Agent)
plan_itinerary = Task(
    description='Create a 3-day travel itinerary for a trip to Tokyo, Japan. Focus on cultural sites and activities.',
    expected_output='A markdown file with a day-by-day plan, including morning, afternoon, and evening activities.',
    agent=travel_agent
)

# Task 2: Recommend Restaurants (for the Food Critic)
# This task uses the output of the first task as its context.
recommend_restaurants = Task(
    description='Based on the planned itinerary, recommend one authentic restaurant for each day of the trip.',
    expected_output='A list of 3 restaurant names, each with a brief description and why it fits the itinerary.',
    agent=food_critic,
    context=[plan_itinerary]
)

# Form the crew with a sequential process
trip_crew = Crew(
    agents=[travel_agent, food_critic],
    tasks=[plan_itinerary, recommend_restaurants],
    process=Process.sequential,
    verbose=True
)

print("--- Kicking off the Trip Planning Crew ---")
trip_result = trip_crew.kickoff()

print("\n--- Final Itinerary ---")
print(trip_result)

--- Kicking off the Trip Planning Crew ---


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 314ead93-74fa-4c93-b6b8-f1d1ab306526                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert Travel Agent                                                                                     │
│                                                                                                                 │
│  Task: Create a 3-day travel itinerary for a trip to Tokyo, Japan. Focus on cultural sites and activities.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert Travel Agent                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As your dedicated luxury travel expert, I have curated a 3-day cultural immersion into the heart of Tokyo.     │
│  This itinerary is designed to balance iconic landmarks with the hidden gems that reveal the city's true soul.  │
│  We will journey from the sacred grounds of ancient temples to the pulsing energy of modern art, all while      │
│  indulging in the unparalleled culinary landscape of Japan. Prepare for a journey that engages all your         │
│  senses.                                                                                                        │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### **Your Exclusive 3-Day Tokyo Cultural Itinerary**                                                          │
│                                                                                                                 │
│  **Essential Pre-Trip Preparations:**                                                                           │
│  *   **Transportation:** Purchase a Suica or Pasmo card upon arrival at the airport. This rechargeable smart    │
│  card is essential for seamless travel on all trains, subways, and buses.                                       │
│  *   **Connectivity:** Rent a pocket Wi-Fi device at the airport or arrange for an eSIM. Staying connected is   │
│  crucial for navigation and on-the-go research.                                                                 │
│  *   **Reservations:** For specific high-end dining experiences or popular activities like the teamLab museum,  │
│  booking in advance is highly recommended.                                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **Day 1: The Soul of Old Tokyo & The Elegance of Ginza**                                                   │
│                                                                                                                 │
│  This day is a tale of two cities: the historic, spiritual heart of Tokyo and its most sophisticated,           │
│  glittering district.                                                                                           │
│                                                                                                                 │
│  **Morning (9:00 AM - 1:00 PM): Asakusa's Ancient Splendor**                                                    │
│                                                                                                                 │
│  *   **Activity:** Begin at **Sensō-ji Temple**, Tokyo's oldest and most significant Buddhist temple. Approach  │
│  through the Kaminarimon (Thunder Gate) and walk down the **Nakamise-dori**, a vibrant market street that has   │
│  served pilgrims for centuries. Here, you can sample tr

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: b774d99a-28a7-45b4-bc43-9e5f008fded1                                                                     │
│  Agent: Expert Travel Agent                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Local Food Critic                                                                                       │
│                                                                                                                 │
│  Task: Based on the planned itinerary, recommend one authentic restaurant for each day of the trip.             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 314ead93-74fa-4c93-b6b8-f1d1ab306526                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Ah, an exquisite itinerary! As a local who has spent years exploring every back alley and        │
│  hidden gem of Tokyo's food scene, I can see the thoughtful curation by your travel expert. While their         │
│  suggestions are excellent, allow me to offer my personal, on-the-ground recommendations for a truly authentic  │
│  taste of our city, one that syncs perfectly with your daily travels.                                           │
│                                                                                                                 │
│  Here are my selections for your three-day culinary journey:                                                    │
│                                                                                                                 │
│  ### **Day 1: Asakusa Unatetsu (浅草 うな鐵)**                                                                  │
│                                                                                                                 │
│  *   **Description:** Nestled in the heart of the historic Asakusa district you'll be exploring, Asakusa        │
│  Unatetsu is a master of one thing: *unagi* (freshwater eel). This isn't just any grilled eel; they are famous  │
│  for their *hitsumabushi*, a Nagoya-style dish where grilled eel is served over rice in a traditional wooden    │
│  container. You enjoy it in three distinct ways: first on its own, then with condiments like wasabi and         │
│  scallions, and finally by pouring a savory dashi broth over it, creating a comforting porridge-like dish. The  │
│  atmosphere is bustling, traditional, and filled with the irresistible aroma of eel grilling over charcoal.     │
│  *   **Why it Fits the Itinerary:** After immersing yourself in the ancient splendor of Sensō-ji Temple,        │
│  dining at Unatetsu continues the "Soul of Old Tokyo" experience. Unagi is a classic Edo-period delicacy, and   │
│  eating it here, just steps from the temple, connects you directly to the culinary history of the area. It      │
│  provides a more specialized and deeply traditional alternative to the bento box suggested for lunch, making    │
│  your first meal a memorable deep dive into Edo cuisine.                                                        │
│                                                                                                                 │
│  ### **Day 2: Menya Musashi (麺屋武蔵)**                                                                        │
│                                                                                                                 │
│  *   **Description:** Forget everything you think you know about ramen. Tucked away in the bustling heart of    │
│  Shinjuku, Menya Musashi is a legendary establishment that revolutionized the Tokyo ramen scene. Named after a  │
│  famous samurai, its "double soup" broth—a rich, complex blend of pork/chicken bones (*tonkotsu*) and seafood   │
│  (*gyokai*)—is iconic. The noodles are thick and chewy, and the massive, tender chunks of braised pork belly    │
│  (*kakuni*) are melt-in-your-mouth perfection. The vibe is loud, energetic, and unapologetically focused on     │
│  the food.                                                     

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: a827b58d-a977-4710-bf42-d11fdec1723d                                                                     │
│  Agent: Local Food Critic                                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 3. Adding Tools to a CrewAI Agent

CrewAI agents can use tools to access external information and capabilities. Instead of relying on pre-built wrappers which can cause dependency issues, we can create our own custom tool by wrapping the Tavily Python client. This is a robust and flexible approach.

In [3]:
from crewai.tools import BaseTool
from tavily import TavilyClient

# 1. Create a custom tool by inheriting from BaseTool
class TavilySearchTool(BaseTool):
    name: str = "TavilySearch"
    description: str = "A tool that can be used to search the web with Tavily for up-to-date information."
    
    def _run(self, query: str) -> str:
        client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        response = client.search(query=query, search_depth="basic")
        return response['results']

# Instantiate our custom tool
search_tool = TavilySearchTool()

# 2. Create an agent and assign the custom tool to it
market_researcher = Agent(
    role='Market Research Analyst',
    goal='Find and summarize the latest news about a specific company.',
    backstory='You are a skilled analyst who can quickly find and synthesize financial news.',
    tools=[search_tool], # Assign the custom tool here
    verbose=True,
    llm=llm
)

# 3. Create a task for the agent
research_company_task = Task(
    description='Find the top 3 news headlines for NVIDIA since September 2, 2025, summarize them and provide the news date',
    expected_output='A numbered list of 3 headlines, each followed by a one-sentence summary.',
    agent=market_researcher
)

# 4. Form a single-agent crew to run the task
research_crew = Crew(
    agents=[market_researcher],
    tasks=[research_company_task],
    verbose=True
)

print("--- Kicking off the Research Crew with Tools ---")
research_result = research_crew.kickoff()

print("\n--- Final Research Summary ---")
print(research_result)

--- Kicking off the Research Crew with Tools ---


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 442f942e-5eaa-4512-bab3-8b7f744fdc63                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Find the top 3 news headlines for NVIDIA since September 2, 2025, summarize them and provide the news    │
│  date                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Analyst                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I am sorry, but I cannot provide news headlines for NVIDIA from September 2, 2025, as that date is in the      │
│  future. News events have not yet occurred, and therefore no information is available.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 92c18668-2ee5-4171-b37b-a0d229fa6925                                                                     │
│  Agent: Market Research Analyst                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Final Research Summary ---
I am sorry, but I cannot provide news headlines for NVIDIA from September 2, 2025, as that date is in the future. News events have not yet occurred, and therefore no information is available.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 442f942e-5eaa-4512-bab3-8b7f744fdc63                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: I am sorry, but I cannot provide news headlines for NVIDIA from September 2, 2025, as that date  │
│  is in the future. News events have not yet occurred, and therefore no information is available.                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 4. Advanced Capability: Hierarchical Process

For more complex workflows, CrewAI offers a `hierarchical` process. In this mode, a manager agent is dynamically nominated to orchestrate the crew, delegating tasks to other agents and synthesizing the final result. This is suitable for problems that require more dynamic coordination.

In [7]:
# Ensure Task, Crew, and Process are available (in case earlier cells haven't been run)
from crewai import Task, Crew, Process

# Re-using the agents from our first example

# The tasks are the same, but we don't need to specify context
plan_itinerary_h = Task(
    description='Create a 3-day travel itinerary for a trip to Rome, Italy. Focus on historical sites.',
    expected_output='A markdown file with a day-by-day plan.',
    agent=travel_agent
)

recommend_restaurants_h = Task(
    description='Recommend one authentic restaurant for each day of the trip to Rome.',
    expected_output='A list of 3 restaurant names with descriptions.',
    agent=food_critic
)

# Form the crew with a hierarchical process
hierarchical_crew = Crew(
    agents=[travel_agent, food_critic],
    tasks=[plan_itinerary_h, recommend_restaurants_h],
    process=Process.hierarchical,  # Use the hierarchical process
    manager_llm=llm # Specify an LLM for the manager agent
)

print("--- Kicking off the Hierarchical Crew ---")
hierarchical_result = hierarchical_crew.kickoff()

print("\n--- Final Hierarchical Result ---")
print(hierarchical_result)

--- Kicking off the Hierarchical Crew ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert Travel Agent                                                                                     │
│                                                                                                                 │
│  Task: What are the must-see historical sites in Rome, Italy for a 3-day itinerary, and what would be an ideal  │
│  sequence for visiting them to maximize historical learning and enjoyment?                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert Travel Agent                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **3-Day Historical Itinerary for Rome, Italy**                                                                 │
│                                                                                                                 │
│  **Day 1: Ancient Rome Unveiled**                                                                               │
│                                                                                                                 │
│  - **Morning: Colosseum and Roman Forum**                                                                       │
│    - Start your journey at the grand Colosseum, the iconic symbol of Rome’s enduring history. Book a            │
│  skip-the-line private guided tour to dive deep into tales of gladiators and epic battles.                      │
│    - Following the Colosseum, transition to the Roman Forum, the epicenter of ancient Roman political life.     │
│  Explore the ruins where Julius Caesar once walked, including the Senate House and the Temple of Saturn.        │
│                                                                                                                 │
│  - **Lunch: Al Gladiatore**                                                                                     │
│    - A charming traditional restaurant near the Colosseum. Enjoy authentic Roman specialties such as Cacio e    │
│  Pepe.                                                                                                          │
│                                                                                                                 │
│  - **Afternoon: Palatine Hill**                                                                                 │
│    - Head to the serene Palatine Hill, where, according to legend, Romulus founded Rome. Discover the ruins of  │
│  imperial palaces and enjoy panoramic views of the city.                                                        │
│                                                                                                                 │
│  - **Evening: Trastevere**                                                                                      │
│    - Wander through the vibrant and historic district of Trastevere. Have dinner in a quaint trattoria and      │
│  soak in the bohemian vibes.                                                                                    │
│                                                                                                                 │
│  **Day 2: Rome's Religious and Artistic Heritage**                                                              │
│                                                                                                                 │
│  - **Morning: Vatican City**                                                                                    │
│    - Begin with a guided tour of the Vatican Museums. Marvel at masterpieces from antiquity to the              │
│  Renaissance, culminating in Michelangelo's Sistine Chapel ceiling.                                             │
│    - Continue to St. Peter’s Basilica; climb to the top of the dome for an astonishing view of the Vatican and  │
│  Rome.                                                                                                          │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Local Food Critic                                                                                       │
│                                                                                                                 │
│  Task: Recommend one authentic restaurant for each day of the trip to Rome, based on the provided itinerary.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Local Food Critic                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  For your 3-day historical itinerary in Rome, here are my recommendations for one authentic restaurant to       │
│  visit each day:                                                                                                │
│                                                                                                                 │
│  **Day 1: Ancient Rome and Trastevere**                                                                         │
│  After exploring the Colosseum and Roman Forum, head over to Trastevere, a charming neighborhood known for its  │
│  authentic Roman cuisine. I recommend dining at **Trattoria Da Enzo al 29**. This is a beloved spot among       │
│  locals and offers classic Roman dishes like carbonara and saltimbocca in a cozy setting. Make sure to reserve  │
│  a table in advance, as it gets quite busy!                                                                     │
│                                                                                                                 │
│  **Day 2: Vatican City and Surroundings**                                                                       │
│  After a day of exploring the Vatican Museums, St. Peter's Basilica, and the Sistine Chapel, unwind with a      │
│  meal at **Ristorante Arlù**. Located near the Vatican, this family-run restaurant offers authentic Roman       │
│  cuisine with a twist. The friendliness of the staff and the quality of dishes like the cacio e pepe and osso   │
│  buco will make it a memorable dining experience.                                                               │
│                                                                                                                 │
│  **Day 3: Baroque Rome and Via Veneto**                                                                         │
│  Conclude your historical exploration with a visit to the magnificent Baroque sites such as the Pantheon and    │
│  Capitoline Hill. For dinner near Via Veneto, I recommend **Girarrosto Fiorentino**, which somewhat defies its  │
│  name by offering Italian cuisine with an emphasis on perfectly grilled meats. The atmosphere is elegant yet    │
│  welcoming making it a great spot to reflect on your Roman adventures while savoring authentic dishes.          │
│                                                                                                                 │
│  Enjoy your culinary and historical journey through Rome!                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Final Hierarchical Result ---
For your 3-day historical itinerary in Rome, here are my recommendations for one authentic restaurant to visit each day:

**Day 1: Ancient Rome and Trastevere**  
After exploring the Colosseum and Roman Forum, head over to Trastevere, a charming neighborhood known for its authentic Roman cuisine. I recommend dining at **Trattoria Da Enzo al 29**. This is a beloved spot among locals and offers classic Roman dishes like carbonara and saltimbocca in a cozy setting. Make sure to reserve a table in advance, as it gets quite busy!

**Day 2: Vatican City and Surroundings**  
After a day of exploring the Vatican Museums, St. Peter's Basilica, and the Sistine Chapel, unwind with a meal at **Ristorante Arlù**. Located near the Vatican, this family-run restaurant offers authentic Roman cuisine with a twist. The friendliness of the staff and the quality of dishes like the cacio e pepe and osso buco will make it a memorable dining experience.

**Day 3: Baroque Rom

## Lab Conclusion

In this lab, you've learned the fundamentals of CrewAI. You've seen how to define agents with distinct roles, create tasks for them to perform, and assemble them into a crew that can work together sequentially or be managed hierarchically.

**Key Takeaways:**
- CrewAI is excellent for problems that can be broken down into distinct roles and responsibilities.
- The `Agent`, `Task`, and `Crew` abstractions provide a clear and intuitive way to structure multi-agent systems.
- Creating custom tools by inheriting from `BaseTool` is a reliable way to add capabilities and avoid dependency conflicts.
- The `hierarchical` process allows for more dynamic, manager-led coordination for complex, non-linear tasks.